# Vision-Based Landslide Forecasting (PyTorch CUDA Accelerated)
## Phase 1: Pre-Event Forecasting Model (Baseline CNN & ResNet50 Transfer Learning)
This notebook trains landslide forecasting models using **PyTorch** with **NVIDIA CUDA GPU acceleration** on your local machine.

In [2]:
import os
import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

# Check CUDA GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 60)
print(f'PyTorch Version: {torch.__version__}')
print(f'Using Device   : {device}')
if device.type == 'cuda':
    print(f'GPU Name       : {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory     : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
else:
    print('WARNING: CUDA GPU not detected.')
print('=' * 60)

ModuleNotFoundError: No module named 'torch'

## 1. Load Dataset and Verify Split
We load `metadata.csv` and use `GroupShuffleSplit` on `landslide_id` to strictly prevent spatial data leakage.

In [ ]:
metadata_path = 'dataset_version_2/metadata.csv'
if not os.path.exists(metadata_path):
    metadata_path = 'metadata.csv'

df = pd.read_csv(metadata_path)
df = df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)
print(f'Total valid images: {len(df)}')

gss1 = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=42)
train_idx, temp_idx = next(gss1.split(df, groups=df['landslide_id']))
df_train = df.iloc[train_idx].copy().reset_index(drop=True)
df_temp  = df.iloc[temp_idx].copy().reset_index(drop=True)

gss2 = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=42)
val_idx, test_idx = next(gss2.split(df_temp, groups=df_temp['landslide_id']))
df_val  = df_temp.iloc[val_idx].copy().reset_index(drop=True)
df_test = df_temp.iloc[test_idx].copy().reset_index(drop=True)

print(f'Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}')

# Positional weight for BCE loss
neg_count = (df_train['label'] == 0).sum()
pos_count = (df_train['label'] == 1).sum()
pos_weight = torch.tensor([neg_count / float(pos_count)], dtype=torch.float32).to(device)
print(f'pos_weight for loss: {pos_weight.item():.3f}')

## 2. PyTorch Dataset and DataLoaders

In [ ]:
class LandslideDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['image_path']
        label = float(self.df.iloc[idx]['label'])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.float32)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_transform = T.Compose([
    T.Resize(IMG_SIZE),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.15, contrast=0.15),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = T.Compose([
    T.Resize(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(LandslideDataset(df_train, train_transform), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(LandslideDataset(df_val, val_test_transform),   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(LandslideDataset(df_test, val_test_transform),  batch_size=BATCH_SIZE, shuffle=False)

## 3. Model 1: Improved Baseline CNN

In [ ]:
class ImprovedBaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.25),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.25),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.25)
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Linear(128, 128), nn.BatchNorm1d(128), nn.ReLU(True), nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

print(ImprovedBaselineCNN())

## 4. Model 2: ResNet50 Transfer Learning

In [ ]:
class ResNet50Transfer(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 128), nn.BatchNorm1d(128), nn.ReLU(True), nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.backbone(x)

print('ResNet50 Transfer Model Created')